# Build a PV pipeline
### Goal: Using clinicalBERT to extract features or fine-tune for NER


## What is d4data/biomedical-near-all
This model is based on DistlBERT and fined-tuned with medical literature and database (MedMentions, BC5CDR). There are 41 Tags and followed UMLS.


### 🏥 Pharmacovigilance (PV) Entity Mapping Logic

| Category | Entity Label | Used in PV? | PV Pillar (Target) |
| :--- | :--- | :---: | :--- |
| **Demographics** | `Age`, `Sex` | ✅ | **Identifiable Patient** |
| **Clinical Findings** | `Sign_symptom`, `Disease_disorder`, `Finding`, `Injury_poisoning` | ✅ | **Adverse Event** |
| **Chemicals & Drugs** | `Medication`, `Pharmacologic_substance`, `Chemical` | ✅ | **Suspected Drug** |
| **Supplements** | `Vitamin` | ⚠️ | Suspected Drug (Optional) |
| **Reporters/Loc** | `Nonbiological_location` | ✅ | **Identifiable Reporter** |
| **Anatomy** | `Anatomy`, `Body_part`, `Tissue`, `Organ`, `Cell`, `Cell_component`, `Gene_or_genome` | ❌ | Hidden (Too detailed) |
| **Procedures** | `Diagnostic_procedure`, `Therapeutic_procedure`, `Laboratory_procedure`, `Health_care_activity` | ❌ | Hidden (Supporting Info) |
| **Biological Ops** | `Biologic_function`, `Mental_process`, `Genetic_function`, `Cell_function`, `Molecular_function` | ❌ | Hidden |
| **Concepts** | `Qualitative_concept`, `Quantitative_concept`, `Temporal_concept`, `Spatial_concept` | ❌ | Hidden |
| **Organisms** | `Organism`, `Bacterium`, `Virus`, `Fungus`, `Archaeon`, `Eukaryote` | ❌ | Hidden |
| **Others** | `Food`, `Entity`, `Medical_device`, `Group`, `Idea_or_concept` | ❌ | Hidden |

In [54]:
# load packages
import numpy as np
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModel


In [55]:
pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.


In [56]:
import transformers
print(transformers.__version__)

5.3.0


In [57]:
#load model and tokenizer
model_name= "emilyalsentzer/Bio_ClinicalBERT"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21473.84it/s]
BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
# load pipelines
from transformers import pipeline

In [68]:
ner_pipeline=pipeline("ner",model="d4data/biomedical-ner-all")
text= "Patient is a 45-year old woman with a history of diabetes and hypertension. She came on 3.21.2025. She presented to the emergency department with chest pain and shortness of breath. On examination, she was found to have elevated blood pressure and an irregular heartbeat. The patient was admitted to the hospital for further evaluation and management. She was admitted today for acute angioedma. The patient reports he recently started Lisinopril 20 mg orally once daily. Three hours after the second dose, she noted sigificantly swelling of the lips and tonuge.In the ED, he was treated with Epinephrine 0.3mg IM, Diphenhydramine 50 mg intravenous push, and Methylprednisolone 125 mg intravenously. The patient was observed for several hours and then discharged home with instructions to avoid Lisinopril and to follow up with his primary care physician in one week."


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 7533.09it/s]


In [69]:
# extract entities
entities=ner_pipeline(text)
print(entities)

#format the output


[{'entity': 'B-Age', 'score': np.float32(0.9996362), 'index': 4, 'word': '45', 'start': 13, 'end': 15}, {'entity': 'I-Age', 'score': np.float32(0.9964071), 'index': 5, 'word': '-', 'start': 15, 'end': 16}, {'entity': 'I-Age', 'score': np.float32(0.9978759), 'index': 6, 'word': 'year', 'start': 16, 'end': 20}, {'entity': 'I-Age', 'score': np.float32(0.99683654), 'index': 7, 'word': 'old', 'start': 21, 'end': 24}, {'entity': 'B-Sex', 'score': np.float32(0.9995958), 'index': 8, 'word': 'woman', 'start': 25, 'end': 30}, {'entity': 'B-Date', 'score': np.float32(0.9132287), 'index': 21, 'word': '3', 'start': 88, 'end': 89}, {'entity': 'I-Date', 'score': np.float32(0.8879411), 'index': 22, 'word': '.', 'start': 89, 'end': 90}, {'entity': 'I-Date', 'score': np.float32(0.96772814), 'index': 23, 'word': '21', 'start': 90, 'end': 92}, {'entity': 'I-Date', 'score': np.float32(0.92949235), 'index': 24, 'word': '.', 'start': 92, 'end': 93}, {'entity': 'I-Date', 'score': np.float32(0.9882615), 'index

In [70]:
# Entity information become readable of human, set aggregate_strategy to "simple"
ner_pipeline_agg=pipeline("ner",model="d4data/biomedical-ner-all", aggregation_strategy="simple")
entities_agg=ner_pipeline_agg(text)
print(entities_agg)
for entity in entities_agg:
    print(f"Entity: {entity['entity_group']}, Label: {entity['entity_group']}, Score: {entity['score']:.2f}, Word: {entity['word']} ")


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 7918.18it/s]


[{'entity_group': 'Age', 'score': np.float32(0.9976889), 'word': '45 - year old', 'start': 13, 'end': 24}, {'entity_group': 'Sex', 'score': np.float32(0.9995958), 'word': 'woman', 'start': 25, 'end': 30}, {'entity_group': 'Date', 'score': np.float32(0.94406694), 'word': '3. 21. 2025', 'start': 88, 'end': 97}, {'entity_group': 'Clinical_event', 'score': np.float32(0.99986196), 'word': 'presented', 'start': 103, 'end': 112}, {'entity_group': 'Nonbiological_location', 'score': np.float32(0.9997836), 'word': 'emergency department', 'start': 120, 'end': 140}, {'entity_group': 'Biological_structure', 'score': np.float32(0.99972695), 'word': 'chest', 'start': 146, 'end': 151}, {'entity_group': 'Sign_symptom', 'score': np.float32(0.99995935), 'word': 'pain', 'start': 152, 'end': 156}, {'entity_group': 'Sign_symptom', 'score': np.float32(0.9993643), 'word': 'shortness of breath', 'start': 161, 'end': 180}, {'entity_group': 'Lab_value', 'score': np.float32(0.8765702), 'word': 'elevated', 'start'

# For the NER pipeline 
score is the confidence score
softmax function and token: the score is 0.99 :take the highest probability as the prediction result


In [44]:
import json

In [71]:
#format the output, extract PV pillers, and handle low confidence entities
# Define a mapping from entity labels to PV pillers
def extract_pv_pillers(ner_results, threshold=0.95):
    
    piller_mapping={"Age":"Identifiable Patient",
             "Sex":"Identifiable Patient",
             "Medication":"Suspected Drug",
             "Sign_symptom": "Adverse Event",
             "Disease_disorder":"Adverse Event",
             "Nonbiological_location":"Identificable Reporter"}
    # Initialize storage for PV pillers, using sets to avoid duplicates
    extract={"Identifiable Patient": set(),
             "Suspected Drug": set(),
             "Adverse Event": set(), 
             "Identificable Reporter": set(),
             "Requires Human Intervention": []} 
    # To avoid duplicate entries in "Requires Human Intervention"
    seen_intervention=set()

    for entity in ner_results:
        label=entity['entity_group']
        score=float(entity['score'])
        word=entity['word'].replace("##","").strip()

    # Handle low confidence entities
        if score< threshold:
            entry_id =(word, label)
            if entry_id not in seen_intervention:
                extract["Requires Human Intervention"].append({"Entity": word, "Label": label, "Score": score})
                seen_intervention.add(entry_id)
            continue
   # Map high confidence entities to PV pillers
    if label in piller_mapping:
        pillar_name=piller_mapping[label]
        if pillar_name=="Identificable Reporter":
            reporter_keywords=["physician","clinic","hospital","department"]
            if any(keyword in word.lower() for keyword in reporter_keywords):           
                extract[pillar_name].add(word)
     
        else:
            extract[pillar_name].add(word)
     # Convert sets to comma-separated strings for final output       
    final_output={k:",".join(list(v)) if isinstance(v,set) else v for k,v in extract.items()}
    return final_output
# Output the extracted PV pillers in JSON format
try:
    result= extract_pv_pillers(entities_agg)
    print(json.dumps(result,indent=4, ensure_ascii=False))
except NameError as e:
    print(f"defined entities_agg variable not found: error message: {e}")
    

{
    "Identifiable Patient": "",
    "Suspected Drug": "",
    "Adverse Event": "",
    "Identificable Reporter": "primary care physician",
    "Requires Human Intervention": [
        {
            "Entity": "3. 21. 2025",
            "Label": "Date",
            "Score": 0.9440669417381287
        },
        {
            "Entity": "elevated",
            "Label": "Lab_value",
            "Score": 0.8765702247619629
        },
        {
            "Entity": "acute",
            "Label": "Detailed_description",
            "Score": 0.7656055688858032
        },
        {
            "Entity": "ed",
            "Label": "Disease_disorder",
            "Score": 0.9140031933784485
        },
        {
            "Entity": "si",
            "Label": "Detailed_description",
            "Score": 0.8933144807815552
        },
        {
            "Entity": "ineph",
            "Label": "Medication",
            "Score": 0.7428653836250305
        },
        {
            "Entity": "predn

In [72]:
# If we set the lower confidence threshold to 0.85, we can see more entities are extracted, but some of them may be less accurate and require human review.
import json

def extract_pv_pillers_dynamic(ner_results, default_threshold=0.90):
    # 1. define dynamic thresholds (confidence requirements)
    # medicine condfident high (0.92)，symptoms and diesase(0.85) to catch more
    threshold_mapping = {
        "Medication": 0.90,
        "Pharmacologic_substance": 0.90,
        "Sign_symptom": 0.85,
        "Disease_disorder": 0.85,
        "Nonbiological_location": 0.85,
        "Age": 0.95,
        "Sex": 0.95
    }

    piller_mapping = {
        "Age": "Identifiable Patient",
        "Sex": "Identifiable Patient",
        "Medication": "Suspected Drug",
        "Pharmacologic_substance": "Suspected Drug",
        "Medicine_start_end": "Suspected Drug",
        "Sign_symptom": "Adverse Event",
        "Disease_disorder": "Adverse Event",
        "Nonbiological_location": "Identificable Reporter"
    }

    extract = {
        "Identifiable Patient": set(),
        "Suspected Drug": set(),
        "Adverse Event": set(), 
        "Identificable Reporter": set(),
        "Requires Human Intervention": []
    }

    seen_intervention = set()

    for entity in ner_results:
        label = entity['entity_group']
        score = float(entity['score'])
        word = entity['word'].replace("##", "").strip()
        
        # 2. get the dynamic threshold for the current label, or use default if not specified
        current_threshold = threshold_mapping.get(label, default_threshold)

        # 3. if the score is below the threshold, flag for human intervention
        if score < current_threshold:
            entry_id = (word, label)
            if entry_id not in seen_intervention:
                extract["Requires Human Intervention"].append({
                    "Entity": word, 
                    "Label": label, 
                    "Score": round(score, 4),
                    "Required_Score": current_threshold  # 提示為什麼被攔截
                })
                seen_intervention.add(entry_id)
            continue

        # 4. based on the mapping, assign high confidence entities to PV pillers
        if label in piller_mapping:
            pillar_name = piller_mapping[label]
            if pillar_name == "Identificable Reporter":
                keywords = ["physician", "clinic", "hospital", "doctor"]
                if any(k in word.lower() for k in keywords):           
                    extract[pillar_name].add(word)
            else:
                extract[pillar_name].add(word)

    #convert sets to comma-separated strings for final output
    return {k: ", ".join(list(v)) if isinstance(v, set) else v for k, v in extract.items()}

# 測試執行
result = extract_pv_pillers_dynamic(entities_agg)
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "Identifiable Patient": "45 - year old, woman",
    "Suspected Drug": "opril, prednisolone, methyl, henhydramine, ep, sin, li, dip",
    "Adverse Event": "pain, shortness of breath, swelling, ed, irregular, ang",
    "Identificable Reporter": "hospital, primary care physician",
    "Requires Human Intervention": [
        {
            "Entity": "elevated",
            "Label": "Lab_value",
            "Score": 0.8766,
            "Required_Score": 0.9
        },
        {
            "Entity": "acute",
            "Label": "Detailed_description",
            "Score": 0.7656,
            "Required_Score": 0.9
        },
        {
            "Entity": "si",
            "Label": "Detailed_description",
            "Score": 0.8933,
            "Required_Score": 0.9
        },
        {
            "Entity": "ineph",
            "Label": "Medication",
            "Score": 0.7429,
            "Required_Score": 0.9
        }
    ]
}


## Not only record the required human intervention, but also display the PV pillar
1. require human intervention
2. passed validation


In [73]:
import json

def extract_pv_pillers_final(ner_results, default_threshold=0.90):
    # 1. 定義動態門檻
    threshold_mapping = {
        "Medication": 0.90,
        "Pharmacologic_substance": 0.90,
        "Sign_symptom": 0.85,
        "Disease_disorder": 0.85,
        "Nonbiological_location": 0.80,
        "Age": 0.95,
        "Sex": 0.95
    }

    piller_mapping = {
        "Age": "Identifiable Patient",
        "Sex": "Identifiable Patient",
        "Medication": "Suspected Drug",
        "Pharmacologic_substance": "Suspected Drug",
        "Medicine_start_end": "Suspected Drug",
        "Sign_symptom": "Adverse Event",
        "Disease_disorder": "Adverse Event",
        "Nonbiological_location": "Identificable Reporter"
    }

    # 初始化容器
    extract = {
        "Identifiable Patient": set(),
        "Suspected Drug": set(),
        "Adverse Event": set(), 
        "Identificable Reporter": set(),
        "Requires Human Intervention": [],
        "Passed_Validation": [] # <--- 新增：記錄成功通過的資訊
    }

    seen_intervention = set()
    seen_passed = set()

    for entity in ner_results:
        label = entity['entity_group']
        score = float(entity['score'])
        word = entity['word'].replace("##", "").strip()
        
        current_threshold = threshold_mapping.get(label, default_threshold)

        # A. 處理低於門檻 (Failed)
        if score < current_threshold:
            entry_id = (word, label)
            if entry_id not in seen_intervention:
                extract["Requires Human Intervention"].append({
                    "Entity": word, "Label": label, "Score": round(score, 4), "Required": current_threshold
                })
                seen_intervention.add(entry_id)
            continue

        # B. 處理高於門檻 (Passed)
        # 紀錄到 Passed 列表
        passed_id = (word, label)
        if passed_id not in seen_passed:
            extract["Passed_Validation"].append({
                "Entity": word, "Label": label, "Score": round(score, 4)
            })
            seen_passed.add(passed_id)

        # C. 歸類到四大支柱
        if label in piller_mapping:
            pillar_name = piller_mapping[label]
            if pillar_name == "Identificable Reporter":
                keywords = ["physician", "clinic", "hospital", "doctor"]
                if any(k in word.lower() for k in keywords):           
                    extract[pillar_name].add(word)
            else:
                extract[pillar_name].add(word)

    # 格式化輸出：將 set 轉為字串
    final_output = {
        k: ", ".join(list(v)) if isinstance(v, set) else v 
        for k, v in extract.items()
    }
    return final_output

# 測試執行
result = extract_pv_pillers_final(entities_agg)
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "Identifiable Patient": "45 - year old, woman",
    "Suspected Drug": "opril, prednisolone, methyl, henhydramine, ep, sin, li, dip",
    "Adverse Event": "pain, shortness of breath, swelling, ed, irregular, ang",
    "Identificable Reporter": "hospital, primary care physician",
    "Requires Human Intervention": [
        {
            "Entity": "elevated",
            "Label": "Lab_value",
            "Score": 0.8766,
            "Required": 0.9
        },
        {
            "Entity": "acute",
            "Label": "Detailed_description",
            "Score": 0.7656,
            "Required": 0.9
        },
        {
            "Entity": "si",
            "Label": "Detailed_description",
            "Score": 0.8933,
            "Required": 0.9
        },
        {
            "Entity": "ineph",
            "Label": "Medication",
            "Score": 0.7429,
            "Required": 0.9
        }
    ],
    "Passed_Validation": [
        {
            "Entity": "45 - year old",
 

In [76]:
# saved the final_resultsas output.JSON file
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)


In [81]:
#read the output file
with open("output.json", "r", encoding="utf-8") as f:
    loaded_result = json.load(f)
print(json.dumps(loaded_result, indent=4, ensure_ascii=False))

{
    "Identifiable Patient": "45 - year old, woman",
    "Suspected Drug": "opril, prednisolone, methyl, henhydramine, ep, sin, li, dip",
    "Adverse Event": "pain, shortness of breath, swelling, ed, irregular, ang",
    "Identificable Reporter": "hospital, primary care physician",
    "Requires Human Intervention": [
        {
            "Entity": "elevated",
            "Label": "Lab_value",
            "Score": 0.8766,
            "Required": 0.9
        },
        {
            "Entity": "acute",
            "Label": "Detailed_description",
            "Score": 0.7656,
            "Required": 0.9
        },
        {
            "Entity": "si",
            "Label": "Detailed_description",
            "Score": 0.8933,
            "Required": 0.9
        },
        {
            "Entity": "ineph",
            "Label": "Medication",
            "Score": 0.7429,
            "Required": 0.9
        }
    ],
    "Passed_Validation": [
        {
            "Entity": "45 - year old",
 

# Pharmacoviligance, establish the adverse effect and specific drug causality
1. Temporal Relationship
2. MedDRA standard codeing (shortness of breath -> Dyspnoes PT code: 10013968, Angioedema - PT code :10002424) Based on the standard code, we can have the Labelledness
3. Causality Algorithms: Naranjo Algorithm (evaluate the adverse side effect)
Naranjo Adverse Drug Reaction (ADR) Probability Scale



### 📊 Naranjo Adverse Drug Reaction (ADR) Probability Scale

| # | Assessment Questions | Yes | No | Don't Know |
| :--- | :--- | :---: | :---: | :---: |
| 1 | Are there previous conclusive reports on this reaction? | **+1** | **0** | **0** |
| 2 | Did the adverse event appear after the suspected drug was administered? | **+2** | **-1** | **0** |
| 3 | Did the ADR improve when the drug was discontinued or an antagonist given? | **+1** | **0** | **0** |
| 4 | Did the ADR reappear when the drug was readministered (Re-challenge)? | **+2** | **-1** | **0** |
| 5 | Are there alternative causes (other than the drug) for the reaction? | **-1** | **+2** | **0** |
| 6 | Did the reaction reappear when a placebo was given? | **-1** | **+1** | **0** |
| 7 | Was the drug detected in the blood in toxic concentrations? | **+1** | **0** | **0** |
| 8 | Was the reaction dose-dependent (more severe when increased)? | **+1** | **0** | **0** |
| 9 | Did the patient have a similar reaction in previous exposure? | **+1** | **0** | **0** |
| 10 | Was the adverse event confirmed by objective evidence (e.g., lab tests)? | **+1** | **0** | **0** |

---

### 📈 Score Interpretation

| Total Score | Probability Category |
| :---: | :--- |
| **≥ 9** | **Definite** ADR |
| **5 – 8** | **Probable** ADR |
| **1 – 4** | **Possible** ADR |
| **≤ 0** | **Doubtful** ADR |

# Mapping to MedDRA

In [74]:
# import difflib, to compare the difference between two outputs, but since we have added more features, the output format has changed significantly, so we will not use it for now.
import difflib

In [ ]:


# 1. Your MedDRA Reference List
MEDDRA_DRUGS = ["Methylprednisolone", "Prednisolone",
"Lisinopril", "Epinephrine", "Diphenhydramine"]
def process_drug_fragments(data):
    # Combine medications from both Passed_Validation and Human_Intervention
    passed_meds = [item['Entity'] for item in data.get("Passed_Validation", []) if item['Label'] == "Medication"]
    manual_meds = [item['Entity'] for item in data.get("Requires Human Intervention", []) if item['Label'] == "Medication"]
    
    # All fragments found in the text
    all_fragments = passed_meds + manual_meds
    
    # Logic: Merge all fragments into a single string to fix tokenization gaps
    # Example: "li" + "sin" + "opril" -> "lisinopril"
    full_text_blob = "".join(all_fragments).replace(" ", "").lower()
    
    mapped_output = []
    
    # 2. Map the blob to MedDRA terms
    for standard_name in MEDDRA_DRUGS:
        std_lower = standard_name.lower()
        
        # Check if the standard name is hidden within our fragments
        # We use a mix of 'in' and fuzzy matching
        if std_lower in full_text_blob or difflib.get_close_matches(std_lower, [full_text_blob], cutoff=0.5):
            mapped_output.append({
                "Standard_Term": standard_name,
                "Status": "✅ Mapped",
                "Source_Fragments": [f for f in all_fragments if f.lower() in std_lower or std_lower in f.lower()]
            })

    return mapped_output



In [ ]:
# --- TEST ---
# (Assuming your JSON is stored in a variable named 'result')
final_results = process_drug_fragments(result)

print("--- Standardized Medication Report ---")
for entry in final_results:
    print(f"Drug: {entry['Standard_Term']} | Status: {entry['Status']}")

--- Standardized Medication Report ---
Drug: Lisinopril | Status: ✅ Mapped
Drug: Diphenhydramine | Status: ✅ Mapped
Drug: Methylprednisolone | Status: ✅ Mapped
Drug: Prednisolone | Status: ✅ Mapped


It still can not work, because epinephrine separte to ep in passed_validation and other words ineph or nephrine in the required human intervention, this word in two categories

In [91]:


def recover_medication_names(json_data, target_drugs):
    # 1. 把所有 Medication 碎片收集起來 (不論它在哪個清單)
    all_fragments = []
    
    # 從 Passed_Validation 抓
    for item in json_data.get("Passed_Validation", []):
        if item["Label"] == "Medication":
            all_fragments.append(item["Entity"])
            
    # 從 Human Intervention 抓 (這就是抓回 'ineph' 或 'nephrite' 的關鍵)
    for item in json_data.get("Requires Human Intervention", []):
        if item["Label"] == "Medication":
            all_fragments.append(item["Entity"])
    
    # 2. 將碎片拼成一個「搜尋字串」
    # 移除空格與特殊字元，變成 "eppinephrite..."
    search_blob = "".join(all_fragments).replace(" ", "").lower()
    
    recovered_list = []
    
    # 3. 在字串中搜尋標準藥名
    for drug in target_drugs:
        drug_lower = drug.lower()
        
        # 使用 difflib 進行模糊比對
        # 即使拼成 "eppinephrite"，它也能識別出非常接近 "epinephrine"
        match = difflib.get_close_matches(drug_lower, [search_blob], n=1, cutoff=0.4)
        
        if match or drug_lower in search_blob:
            recovered_list.append({
                "Standard_Name": drug,
                "Match_Status": "Passed"
            })
            
    return recovered_list



#Test


The Epinephrine does not match, but the Prednisolone did not exist

In [92]:
#Revised codes
target_list = ["Lisinopril", "Epinephrine", "Diphenhydramine", "Methylprednisolone", "Prednisolone"]
print(recover_medication_names(result, target_list))

[{'Standard_Name': 'Lisinopril', 'Match_Status': 'Passed'}, {'Standard_Name': 'Diphenhydramine', 'Match_Status': 'Passed'}, {'Standard_Name': 'Methylprednisolone', 'Match_Status': 'Passed'}, {'Standard_Name': 'Prednisolone', 'Match_Status': 'Passed'}]


In [83]:


# 1. MedDRA 參考清單 (注意：順序很重要，長單字應放在短單字前面，避免誤判)
MEDDRA_DRUGS = ["Methylprednisolone", "Lisinopril", "Epinephrine", "Diphenhydramine", "Prednisolone"]

def process_drug_fragments_revised(data):
    # 同時從兩個來源提取標籤為 Medication 的實體
    passed_meds = [item['Entity'] for item in data.get("Passed_Validation", []) if item['Label'] == "Medication"]
    manual_meds = [item['Entity'] for item in data.get("Requires Human Intervention", []) if item['Label'] == "Medication"]
    
    # 所有的原始碎片 (例如: ['li', 'sin', 'opril', 'ep', 'ineph', ...])
    all_fragments = passed_meds + manual_meds
    
    # 關鍵修正：將碎片拼成一個帶有空格的字串，增加模糊比對的準確度
    # 這樣 "ep" 和 "ineph" 會被視為相鄰或整體的特徵
    full_text_blob = "".join(all_fragments).replace(" ", "").lower()
    
    mapped_output = []
    found_std_terms = set() # 避免重複映射

    # 2. 依照標準藥名進行比對
    for standard_name in MEDDRA_DRUGS:
        std_lower = standard_name.lower()
        
        # 邏輯 A：直接包含判斷 (處理拼接後的碎片)
        # 邏輯 B：模糊比對 (處理拼字錯誤或漏字)
        is_match = std_lower in full_text_blob or \
                   len(difflib.get_close_matches(std_lower, [full_text_blob], cutoff=0.6)) > 0
        
        if is_match:
            # 關鍵修正：防止 "Methylprednisolone" 被同時識別成 "Prednisolone"
            # 如果已經匹配了較長的藥名，就跳過其子集合
            if any(standard_name in already for already in found_std_terms):
                continue

            # 找出組成該藥名的碎片 (用於 Traceability)
            source_frags = [f for f in all_fragments if f.lower() in std_lower]
            
            mapped_output.append({
                "Standard_Term": standard_name,
                "Status": "✅ Mapped",
                "Confidence": "High" if std_lower in full_text_blob else "Medium",
                "Source_Fragments": list(set(source_frags)) # 去重
            })
            found_std_terms.add(standard_name)

    return mapped_output



In [86]:
# --- TEST ---
# (Assuming your JSON is stored in a variable named 'result')
final_results = process_drug_fragments_revised(result)

print("--- Standardized Medication Report ---")
for entry in final_results:
    print(f"Drug: {entry['Standard_Term']} | Status: {entry['Status']}")

--- Standardized Medication Report ---
Drug: Methylprednisolone | Status: ✅ Mapped
Drug: Lisinopril | Status: ✅ Mapped
Drug: Diphenhydramine | Status: ✅ Mapped
Drug: Prednisolone | Status: ✅ Mapped


In [ ]:
# Prepare the MedDRA mapping dictionary
MedDRA_drug=["Lisinopril", "Epinephrine", "Diphenhydramine", "Methylprednisolone"]
# extract the Medication from the ouput of Passed_Validation, if yes, found it and map to MedDRA, if not found, then we can check the Requires Human Intervention list to see if there are any medications that need to be reviewed by human.
def clean_and_map medication(Passed_Validation, Required_Human_Intervention):
    #extraction the medication from passed and human intervention, then map to MedDRA
    all_meds=[item for item in Passed_Validation if item['Label'] in ['Medication', 'Pharmacologic_substance']]
    manual_meds=[item for item in Required_Human_Intervention if item['Label'] in ['Medication', 'Pharmacologic_substance']]
    
    combined_raws="".join[m['Entity'] for m in all_meds]).replace(" ", "").lower()
    mapped_results= []
    seen_drugs=set()

# Check for MedDRA matches in the combined string
    for standard_drug in MedDRA_drug:
        if standard_drug.lower() in combined_raws or  \
           difflib.get_close_matches(combined_raws, [standard_drug.lower()], n=1, cutoff=0.4):
            if standard_drug not in seen_drugs:
                mapped_results.append({"Detected_Fragment": combined_raws, "MedDRA_Mapped": standard_drug,"Status": "Auto_mapped"})
                seen_drugs.add(standard_drug)
                



In [75]:


# 1. 準備 MedDRA 標準藥名字典 (建議使用首字母大寫)
MedDRA_drug = ["Lisinopril", "Epinephrine", "Diphenhydramine", "Methylprednisolone", "Prednisolone"]

def clean_and_map_medication(passed_validation, manual_intervention):
    # 提取所有藥物相關實體 (包含通過與待審核的)
    all_meds = [item for item in passed_validation if item['Label'] == "Medication"]
    manual_meds = [item for item in manual_intervention if item['Label'] == "Medication"]
    
    # 合併所有的碎片文字 (例如: li + sin + opril -> lisinopril)
    # 在醫學 NLP 中，通常會將相鄰的片段串接起來
    combined_raw_text = "".join([m['Entity'] for m in all_meds]).replace(" ", "").lower()
    
    mapped_results = []
    seen_drugs = set()

    # 2. 遍歷標準庫，檢查合併後的文字是否包含這些藥名 (模糊比對)
    for standard_drug in MedDRA_drug:
        # 使用 difflib 尋找最接近的匹配
        # 或者使用簡單的字串包含判斷 (針對碎片化嚴重的狀況)
        if standard_drug.lower() in combined_raw_text or \
           difflib.get_close_matches(combined_raw_text, [standard_drug.lower()], n=1, cutoff=0.4):
            
            if standard_drug not in seen_drugs:
                mapped_results.append({
                    "Detected_Fragments": combined_raw_text,
                    "MedDRA_Standard_Name": standard_drug,
                    "Status": "✅ Auto-Mapped"
                })
                seen_drugs.add(standard_drug)

    # 3. 處理需要人工干預 (Requires Human Intervention) 的孤立碎片
    for manual in manual_meds:
        frag = manual['Entity']
        # 如果這個碎片沒被包含在已識別的藥物中，則標記為待審核
        if not any(frag in res['Detected_Fragments'] for res in mapped_results):
            mapped_results.append({
                "Detected_Fragments": frag,
                "MedDRA_Standard_Name": "PENDING REVIEW",
                "Status": "⚠️ Manual Intervention Required",
                "Score": manual['Score']
            })

    return mapped_results

# 執行範例
final_meds = clean_and_map_medication(result["Passed_Validation"], result["Requires Human Intervention"])